# HealthyLife Policy Assistant — RAG Demo

**A. Indexing:** PDF → Chunking → Embedding → MongoDB Atlas Vector Search
**B. Query:** Question → Query Embedding → Vector Search → Relevant Chunks → Prompt + Context → LLM (Groq · Qwen) → Answer

Setup: `pip install -r requirements.txt`, copy `.env.example` to `.env` and fill in `MONGODB_URI` and `GROQ_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # re-read .env even if a var was already set (e.g. empty)
MONGODB_URI = os.environ["MONGODB_URI"]
PDF_PATH = "data/healthylife_policy_handbook.pdf"
DB_NAME, COLLECTION, INDEX_NAME = "healthylife_rag", "policy_chunks", "vector_index"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "qwen/qwen3.8-27b"

## A. Indexing pipeline
### A1. Load the PDF (one Document per page)

In [ ]:
%pip install -q pypdf langchain-core
from pypdf import PdfReader
from langchain_core.documents import Document

pages = [
    Document(page_content=p.extract_text(), metadata={"source": PDF_PATH, "page": i + 1})
    for i, p in enumerate(PdfReader(PDF_PATH).pages)
]
print(f"{len(pages)} pages, {sum(len(p.page_content) for p in pages)} characters")

### A2. Chunking
Recursive splitting tries paragraph → line → sentence → word boundaries, so chunks stay readable. The overlap keeps sentences that sit on a boundary retrievable.

In [ ]:
%pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks = splitter.split_documents(pages)
print(f"{len(chunks)} chunks")
print("--- sample chunk ---")
print(chunks[4].page_content)

### A3. Embedding
`all-MiniLM-L6-v2` runs locally (no API key) and turns text into a 384-dimensional vector.

In [ ]:
%pip install -q langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})
DIMS = len(embeddings.embed_query("hello"))
print("embedding dimensions:", DIMS)

### A4. Store in MongoDB and create the vector index
Each chunk becomes one document: `text`, `embedding`, `page`, `source`. Re-running this cell replaces the old chunks.

In [ ]:
%pip install -q pymongo langchain-mongodb
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch

client = MongoClient(MONGODB_URI)
collection = client[DB_NAME][COLLECTION]
vector_store = MongoDBAtlasVectorSearch(
    collection=collection, embedding=embeddings, index_name=INDEX_NAME, relevance_score_fn="cosine"
)

collection.delete_many({})
vector_store.add_documents(chunks)
print(collection.count_documents({}), "chunks stored in", f"{DB_NAME}.{COLLECTION}")

if not any(ix["name"] == INDEX_NAME for ix in collection.list_search_indexes()):
    vector_store.create_vector_search_index(dimensions=DIMS, wait_until_complete=120)
    print("created vector index", INDEX_NAME)
else:
    print("vector index", INDEX_NAME, "already exists")

# Atlas Search indexes new writes asynchronously: wait until every chunk is searchable
import time
for _ in range(60):
    if len(vector_store.similarity_search("policy", k=len(chunks))) == len(chunks):
        break
    time.sleep(1)
else:
    raise TimeoutError("vector index did not catch up within 60 s")
print("all chunks searchable")

## B. Query pipeline — step by step
### B1. Question → query embedding

In [ ]:
question = "Can I get a refund if I cancel my annual plan after two months?"
query_vector = embeddings.embed_query(question)
print(len(query_vector), "dims:", [round(x, 3) for x in query_vector[:6]], "...")

### B2. Vector search → relevant chunks
The raw `$vectorSearch` aggregation that the LangChain retriever runs under the hood:

In [ ]:
results = collection.aggregate([
    {"$vectorSearch": {
        "index": INDEX_NAME, "path": "embedding", "queryVector": query_vector,
        "numCandidates": 100, "limit": 4,
    }},
    {"$project": {"_id": 0, "text": 1, "page": 1, "score": {"$meta": "vectorSearchScore"}}},
])
for r in results:
    print(f"score={r['score']:.3f}  page={r['page']}  {r['text'][:110]!r}...")

### B3. Prompt + context → LLM → answer
Groq serves Qwen. `reasoning_format="hidden"` drops Qwen's `<think>` output so only the final answer is returned.

In [ ]:
%pip install -q langchain-groq

from getpass import getpass
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the HealthyLife policy assistant. Answer using ONLY the context below. "
     "If the answer is not in the context, say you don't know. "
     "Keep it short and cite pages like (p. 3). "
     "Reply in plain text without Markdown; use '- ' for list items.\n\nContext:\n{context}"),
    ("human", "{question}"),
])
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")
llm = ChatGroq(model=LLM_MODEL, temperature=0, reasoning_format="hidden")
retriever = vector_store.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    return "\n\n".join(f"[p. {d.metadata['page']}] {d.page_content}" for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
print(rag_chain.invoke(question))

## Try more questions

In [ ]:
for q in [
    "How many days of paid annual leave do employees get?",
    "What happens if I miss two sessions in a month?",
    "How long is health data kept after I close my account?",
    "What certifications does a coach need?",
    "Does HealthyLife offer a free trial for dogs?",  # not in the handbook -> should say it doesn't know
]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q), "\n")

## Retrieval sanity check
Fails if the right chunk does not reach the top 4, e.g. after changing chunk size or embedding model.

In [ ]:
checks = {
    "How many days of paid annual leave do employees get?": "22 days",
    "How much is the monthly wellness stipend?": "USD 75",
    "How soon must an injury during a session be reported?": "24 hours",
    "What is the late cancellation window for sessions?": "24 hours before",
}
for q, fact in checks.items():
    top = vector_store.similarity_search(q, k=4)
    assert any(fact in d.page_content for d in top), f"missed: {q}"
print("all retrieval checks passed")